In [2]:
! pip install findspark

# Ejercicio Práctico: Análisis de Datos de Fútbol (Transfermarkt)

**Contexto:** El conjunto de datos está basado en el dataset *Football Data from Transfermarkt* de Kaggle. Se compone de varios archivos CSV con información detallada sobre competiciones, juegos, clubes, jugadores y apariciones de partidos.

---

### Inciso 1: Países con mayor número de jugadores
Determine los tres países con mayor número de jugadores (jugadores nacidos en ese país). El resultado debe estar ordenado de forma descendente.

---

### Inciso 2: Jugadores con tarjeta roja
Obtenga la lista de jugadores con tarjeta roja. La salida debe contener dos columnas: el nombre de pila del jugador y la cantidad de tarjetas rojas que tiene.

---

### Inciso 3: Cantidad de juegos en la Premier League
¿Cuántos juegos se jugaron en la Premier League? La salida debe contener dos columnas: el nombre de la liga y la cantidad de juegos que se jugaron en ella.

---

### Inciso 4: Ligas con mayor asistencia de público
Obtenga las tres ligas con mayor número de asistencia de público teniendo en cuenta todos los juegos que se jugaron en ellas. El resultado debe estar ordenado de forma descendente y tener dos columnas: el nombre de la liga y la asistencia total.

In [5]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio Analisis Futbol Transfermarket").getOrCreate()

## Lectura de archivos

In [8]:
players = spark.read.option("header", "true").option("inferSchema", "true").csv("./data/players.csv")

app = spark.read.option("header", "true").option("inferSchema", "true").csv("./data/appearances.csv")

competicion = spark.read.option("header", "true").option("inferSchema", "true").csv("./data/competitions.csv")

juegos = spark.read.option("header", "true").option("inferSchema", "true").csv("./data/games.csv")

## Visualizacion de archivos

In [9]:
players.printSchema()

root
 |-- player_id: integer (nullable = true)
 |-- current_club_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- pretty_name: string (nullable = true)
 |-- country_of_birth: string (nullable = true)
 |-- country_of_citizenship: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- position: string (nullable = true)
 |-- sub_position: string (nullable = true)
 |-- foot: string (nullable = true)
 |-- height_in_cm: integer (nullable = true)
 |-- market_value_in_gbp: double (nullable = true)
 |-- highest_market_value_in_gbp: double (nullable = true)
 |-- url: string (nullable = true)



In [10]:
players.show()

+---------+---------------+--------------------+--------------------+----------------+----------------------+-------------+----------+------------------+-----+------------+-------------------+---------------------------+--------------------+
|player_id|current_club_id|                name|         pretty_name|country_of_birth|country_of_citizenship|date_of_birth|  position|      sub_position| foot|height_in_cm|market_value_in_gbp|highest_market_value_in_gbp|                 url|
+---------+---------------+--------------------+--------------------+----------------+----------------------+-------------+----------+------------------+-----+------------+-------------------+---------------------------+--------------------+
|    38790|          28095|      dmitri-golubov|      Dmitri Golubov|           UdSSR|                Russia|   1985-06-24|    Attack|    Centre-Forward| Both|         178|               NULL|                   675000.0|https://www.trans...|
|   106539|          28095|  ale

In [11]:
from pyspark.sql.functions import col, desc

In [21]:
players.groupBy("country_of_birth").count().orderBy(desc("count")).filter(col("country_of_birth").isNotNull()).limit(3).show()

+----------------+-----+
|country_of_birth|count|
+----------------+-----+
|          France| 1694|
|           Spain| 1388|
|           Italy| 1312|
+----------------+-----+



# Segundo paso

In [22]:
from pyspark.sql.functions import sum, desc

(app.groupBy('player_id')
    .agg(sum(col('red_cards')).alias('total_rojas'))
    .orderBy(col('total_rojas').desc())
    .join(players, ['player_id'], 'left')
    .select(
        col('pretty_name'),
        col('total_rojas')
    )
).show()

+--------------------+-----------+
|         pretty_name|total_rojas|
+--------------------+-----------+
|    Marco Sportiello|          1|
|         Ivan Martic|          0|
|        Rune Hastrup|          0|
| Mehmet Enes Sigirci|          0|
|     Damien Da Silva|          4|
|       Thomas Phibel|          0|
|  Stelios Pozatzidis|          0|
|       Aleandro Rosi|          0|
|Konstantinos Kats...|          0|
|        Ferhat Kiraz|          0|
|        Kaya Tarakci|          0|
|        Igor Koshman|          0|
|          Tarik Kada|          0|
|           Juan Cala|          0|
|    Ferran Corominas|          0|
|    Fernando Pacheco|          1|
|        Wesley Hoedt|          1|
|         Diego Alves|          0|
|          Matt Gilks|          0|
|        Gerard Pique|          2|
+--------------------+-----------+
only showing top 20 rows



```python
(app.groupBy('player_id')

```

* **Agrupación:** Toma el DataFrame llamado `app` y agrupa los registros basándose en la columna `player_id`. A partir de aquí, cualquier operación se aplicará a nivel de cada jugador.

```python
    .agg(sum(col('red_cards')).alias('total_rojas'))

```

* **Agregación:** Suma todos los valores de la columna `red_cards` para cada `player_id`. Con `.alias('total_rojas')`, renombra el resultado de esa suma para que la nueva columna se llame `total_rojas`.

```python
    .orderBy(col('total_rojas').desc())

```

* **Ordenamiento:** Ordena el DataFrame resultante de manera descendente (`desc()`) utilizando la nueva columna `total_rojas`. Es decir, los jugadores con más tarjetas rojas aparecerán arriba.

```python
    .join(players, ['player_id'], 'left')

```

* **El Cruce (Join):** Aquí es donde se realiza la combinación de datos.
* Se cruza el resultado actual con un segundo DataFrame llamado `players`.
* La condición de cruce es la columna común `['player_id']`.
* **Tipo de Join:** especifica tipo **`'left'` join** (Left Outer Join). Esto significa que se mantendrán *todos* los jugadores que venían del DataFrame `app`, incluso si no encuentran una coincidencia exacta en el DataFrame `players` (en cuyo caso, sus datos de `players` vendrán como `null`).



```python
    .select(
        col('pretty_name'),
        col('total_rojas')
    )

```

* **Selección de Columnas:** Finalmente, limpia el resultado final para quedarse únicamente con dos columnas en la vista: `pretty_name` (que viene del DataFrame `players`) y `total_rojas` (calculada previamente).

---

### 3. Salida en Consola

```python
).show()

```

* **Acción:** En PySpark, todas las operaciones anteriores son *transformaciones lazys* (evaluación perezosa). El método `.show()` es la **acción** que le ordena a PySpark ejecutar todo el plan y mostrar los primeros 20 resultados en una tabla por consola.

# Tercer Paso

In [23]:
competicion.show()

+--------------+--------------------+------------------+----------+------------+--------------------+-------------+--------------------+
|competition_id|                name|              type|country_id|country_name|domestic_league_code|confederation|                 url|
+--------------+--------------------+------------------+----------+------------+--------------------+-------------+--------------------+
|            L1|          bundesliga|        first_tier|        40|     Germany|                  L1|       europa|https://www.trans...|
|           DFB|           dfb-pokal|      domestic_cup|        40|     Germany|                  L1|       europa|https://www.trans...|
|           DFL|        dfl-supercup|domestic_super_cup|        40|     Germany|                  L1|       europa|https://www.trans...|
|           NL1|          eredivisie|        first_tier|       122| Netherlands|                 NL1|       europa|https://www.trans...|
|           NLP|     toto-knvb-beker|    

In [24]:
juegos.show()

+-------+----------------+------+-------------+----------+------------+------------+---------------+---------------+------------------+------------------+--------------------+----------+--------------------+--------------------+
|game_id|competition_code|season|        round|      date|home_club_id|away_club_id|home_club_goals|away_club_goals|home_club_position|away_club_position|             stadium|attendance|             referee|                 url|
+-------+----------------+------+-------------+----------+------------+------------+---------------+---------------+------------------+------------------+--------------------+----------+--------------------+--------------------+
|2457642|            NLSC|  2014|        Final|2014-08-03|        1269|         610|              1|              0|              NULL|              NULL| Johan Cruijff ArenA|     42000|      Danny Makkelie|https://www.trans...|
|2639088|            BESC|  2013|        Final|2014-07-20|          58|         498|

In [25]:
# como los datos que nos piden se encuentran en df diferentes hay que hacer un join para concentrar todo en un df y poder responder la pregunta que nos solicitan

data1 = juegos.join(competicion, col("competition_id") == col("competition_code"), "left")



In [26]:
data1.show()

+-------+----------------+------+-------------+----------+------------+------------+---------------+---------------+------------------+------------------+--------------------+----------+--------------------+--------------------+--------------+--------------------+------------------+----------+------------+--------------------+-------------+--------------------+
|game_id|competition_code|season|        round|      date|home_club_id|away_club_id|home_club_goals|away_club_goals|home_club_position|away_club_position|             stadium|attendance|             referee|                 url|competition_id|                name|              type|country_id|country_name|domestic_league_code|confederation|                 url|
+-------+----------------+------+-------------+----------+------------+------------+---------------+---------------+------------------+------------------+--------------------+----------+--------------------+--------------------+--------------+--------------------+--------

In [28]:
data1.groupBy("name").count().filter(col("name") == "premier-league").show()

+--------------+-----+
|          name|count|
+--------------+-----+
|premier-league| 2809|
+--------------+-----+



# Cuarto Paso

In [30]:
from pyspark.sql.functions import sum

In [33]:
from pyspark.sql.functions import sum, desc, col

(data1.groupBy("name")
    .agg(sum("attendance").alias("asitencia"))
    .orderBy(col("asitencia").desc())
    .show()
)

+--------------------+---------+
|                name|asitencia|
+--------------------+---------+
|      premier-league| 86964852|
|          bundesliga| 78102473|
|              laliga| 62943533|
|             serie-a| 53475147|
|             ligue-1| 51593963|
|uefa-champions-le...| 35154225|
|          eredivisie| 34572418|
|       europa-league| 28710888|
|        premier-liga| 25823581|
|  liga-portugal-bwin| 20072843|
|  jupiler-pro-league| 17817099|
|           super-lig| 17455236|
|scottish-premiership| 17379753|
|europa-league-qua...| 12810167|
|uefa-champions-le...|  9479701|
|             efl-cup|  9162166|
|           dfb-pokal|  8404075|
|         superligaen|  7945555|
|        copa-del-rey|  7052640|
|      super-league-1|  6417136|
+--------------------+---------+
only showing top 20 rows


In [34]:
# Las tres mayores ligas fueron la premier-league| - bundesliga y laliga|